# Random Forest Training on Google Colab

This notebook trains a Random Forest model for CoStar commercial real estate data.

**Training Configuration:**
- Dataset: 7,520 training samples × 195 features
- Model: Random Forest with GridSearchCV hyperparameter tuning
- Training time: 2-3 hours on CPU, 30-45 min on GPU
- Memory usage: ~500MB-1GB (well within Colab's 12GB)

**Steps:**
1. Setup environment
2. Upload files to Colab
3. Run training pipeline
4. Download trained model

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install xgboost lightgbm feature-engine --quiet

# Check GPU availability
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Check available RAM
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Total RAM: {ram_gb:.1f} GB")

# If you want to enable GPU for scikit-learn (experimental)
# !pip install cuml-cu11 --quiet

## 2. Upload Files to Colab

**Option A: Upload manually (for small files)**

In [ ]:
# Upload files manually
from google.colab import files

# Upload your project files
# You'll need to upload:
# - train_cloud.py
# - src/ directory (as zip)
# - data/ directory with train.csv, val.csv, test.csv (as zip)

uploaded = files.upload()

# Unzip if you uploaded zip files
!unzip -q src.zip
!unzip -q data.zip

**Option B: Use Google Drive (recommended for larger files)**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy project from Drive
# First, upload your project to Google Drive at: My Drive/MIS587FinalProject/
!cp -r /content/drive/MyDrive/MIS587FinalProject/* /content/

# Verify files
!ls -la
!ls -la src/
!ls -la data/

## 3. Verify Data

In [ ]:
import pandas as pd

# Load and verify data
train = pd.read_csv('../data/train.csv')
val = pd.read_csv('../data/val.csv')
test = pd.read_csv('../data/test.csv')

print(f"Train: {train.shape}")
print(f"Val:   {val.shape}")
print(f"Test:  {test.shape}")

# Check target variable
print(f"\nTarget variable (Rent/SF/Yr):")
print(f"  Mean: ${train['Rent/SF/Yr'].mean():.2f}")
print(f"  Median: ${train['Rent/SF/Yr'].median():.2f}")
print(f"  Missing: {train['Rent/SF/Yr'].isna().sum()} / {len(train)}")

## 4. Train Model (Quick Test)

First, run a quick test without hyperparameter tuning (~30 seconds)

In [ ]:
from train_cloud import train_and_save_model

# Quick test run (no hyperparameter tuning)
print("Running quick test (no hyperparameter tuning)...")
results = train_and_save_model(
    tune_hyperparameters=False,  # Fast mode
    cv_folds=3  # Fewer folds for speed
)

print("\n" + "="*70)
print("QUICK TEST COMPLETE!")
print("="*70)
print(f"Validation R²: {results['val_metrics']['R²']:.3f}")
print(f"Test R²: {results['test_metrics']['R²']:.3f}")

## 5. Train Model (Full Hyperparameter Tuning)

Now run the full training with GridSearchCV (2-3 hours)

In [ ]:
from train_cloud import train_and_save_model
import time

start_time = time.time()

print("Starting full training with hyperparameter tuning...")
print("Expected time: 2-3 hours on CPU, 30-45 min on GPU")
print("")

results = train_and_save_model(
    tune_hyperparameters=True,  # Full GridSearchCV
    cv_folds=5  # 5-fold cross-validation
)

elapsed_time = time.time() - start_time
print(f"\nTotal training time: {elapsed_time/3600:.2f} hours")

print("\n" + "="*70)
print("FULL TRAINING COMPLETE!")
print("="*70)
print(f"Best hyperparameters: {results['model']['model'].get_params()}")
print(f"\nValidation Performance:")
print(f"  R²:   {results['val_metrics']['R²']:.3f}")
print(f"  MAE:  ${results['val_metrics']['MAE']:.2f}")
print(f"  MAPE: {results['val_metrics']['MAPE']:.1f}%")
print(f"\nTest Performance:")
print(f"  R²:   {results['test_metrics']['R²']:.3f}")
print(f"  MAE:  ${results['test_metrics']['MAE']:.2f}")
print(f"  MAPE: {results['test_metrics']['MAPE']:.1f}%")

## 6. Inspect Results

In [ ]:
# Show top features
print("Top 20 Most Important Features:")
print(results['importance_df'].to_string(index=False))

# Plot feature importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(
    results['importance_df']['feature'][:20],
    results['importance_df']['importance'][:20]
)
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Show saved file paths
print("\nSaved files:")
print(f"  Model: {results['paths']['model']}")
print(f"  Metadata: {results['paths']['metadata']}")

## 7. Download Trained Model

In [ ]:
from google.colab import files

# Download model files
files.download(results['paths']['model'])
files.download(results['paths']['metadata'])

print("\n✓ Model files downloaded!")
print("\nNext steps:")
print("1. Place downloaded files in your local ./models/ directory")
print("2. Use predict.py to make predictions on new data")
print("\nExample:")
print("from predict import CoStarPredictor")
print("predictor = CoStarPredictor(")
print("    model_path='./models/rf_model_YYYYMMDD_HHMMSS.pkl',")
print("    metadata_path='./models/rf_metadata_YYYYMMDD_HHMMSS.json'")
print(")")
print("predictions = predictor.predict(data)")

## 8. Optional: Save to Google Drive

In [ ]:
# Alternative: Save to Google Drive instead of downloading
!mkdir -p /content/drive/MyDrive/MIS587FinalProject/models/
!cp {results['paths']['model']} /content/drive/MyDrive/MIS587FinalProject/models/
!cp {results['paths']['metadata']} /content/drive/MyDrive/MIS587FinalProject/models/

print("✓ Model files saved to Google Drive!")